<center><p float="center">
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<center><font size=10>Applied Generative AI and Agentic AI</center></font>
<center><font size=6>Evaluating Generative AI Workflows</center></font>

# Problem Statement

## Business Context

Nima is a global sportswear brand that recently launched a new line of sneakers. While the launch generated strong early sales and high online interest, the company also observed a noticeable increase in product returns.

Customer ratings alone are not sufficient to understand why customers are dissatisfied. Valuable insights are often hidden inside detailed customer reviews shared across e-commerce platforms, social media, and forums. These reviews frequently mention important product aspects such as sizing, comfort, durability, color, design, and price/value. However:

- Manually reviewing thousands of reviews is slow, expensive, and inconsistent.  
- Different individuals may interpret the same feedback differently, leading to inconsistent analysis.  
- Important patterns such as “runs small” or “sole wears out quickly” may be missed or identified too late.  
- The lack of structured feedback makes it difficult to identify the exact reasons behind customer dissatisfaction and product returns.  

To address this problem, Nima wants to use Aspect-Based Sentiment Analysis (ABSA) to automatically identify product aspects mentioned in customer reviews and determine the sentiment associated with each aspect.

## Objective


Build an LLM-powered proof of concept that processes customer reviews related to the sneaker line and:

- Identifies the primary product aspect mentioned in each review (e.g., sizing, comfort, durability, color, design, price/value).  
- Classifies the sentiment associated with that aspect as Positive, Negative, or Neutral.  

The goal is to move beyond simply knowing that dissatisfaction exists and instead understand which product aspects are driving positive or negative customer feedback. These insights can help the business improve product quality, reduce return rates, and enhance customer satisfaction.

## Data Description

The dataset consists of customer reviews collected for the Nima sneaker line. Each record represents real-world feedback shared by users after using the product, capturing their experience, expectations, and specific issues they faced.

Dataset Structure: Each record in the dataset contains the following fields:

- **text**: The customer review text that will be analyzed. Each review is written in natural language and typically highlights one specific aspect of the product along with the user’s experience.
- **aspect**: The actual product aspect discussed in the review. This is one of the predefined categories: sizing, comfort, durability, color, design, price/value
- **sentiment**: The ground-truth sentiment associated with the identified aspect. It can be one of the following: Positive, Negative, or Neutral



# Installing and Importing Necessary Libraries and Dependencies

In [ ]:
!uv pip install -q \
    pandas==3.0.3 \
    numpy==2.4.4 \
    matplotlib==3.10.9 \
    scikit-learn==1.8.0 \
    openai==2.36.0 \
    deepeval==4.0.2 \
    tabulate==0.9.0

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel, and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# Standard library
import os
import json
import re
import warnings
from collections import Counter

# Data & visualisation
import pandas as pd
import numpy as np
from tabulate import tabulate

# Scikit-learn utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# OpenAI client
from openai import OpenAI

# DeepEval
from deepeval.dataset import Golden
from deepeval.prompt import Prompt
from deepeval.optimizer import PromptOptimizer
from deepeval.optimizer.algorithms import GEPA
from deepeval.models import GPTModel
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval import evaluate

warnings.filterwarnings("ignore")

/tmp/ipykernel_2028/2720364427.py:27: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


### OpenAI API Calling



We will load the API credentials from a JSON configuration file, store them as environment variables, and initialize the OpenAI client for making API requests.

In [ ]:
# Load the JSON file and extract values
file_name = 'config.json'                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    OPENAI_API_KEY = config.get("API_KEY")                               # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")                             # Extract the OpenAI base URL from the config

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY                                   # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE                                 # Set API base URL as environment variable

# Initialize OpenAI client
client = OpenAI()                                                               # Create an instance of the OpenAI client

Sends a user and system prompt to the LLM model and returns the generated text response from the assistant.


In [ ]:
# ──  OpenAI Chat Completions ───────────────────────────────────────────────────
def llm_response(system: str, user: str, temperature: float = 0.1) -> str:
    """Single-turn chat completion. Returns the assistant content string."""
    # Create a completion using the initialized client and global MODEL variable
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=temperature,
        messages=[
            {"role": "system", "content": system}, # Sets the AI persona/rules
            {"role": "user",   "content": user},   # The specific user prompt
        ],
    )
    # Extract and return the text content from the first response choice
    return response.choices[0].message.content

Test the Chat Completions API using a simple query and observe the generated response.

In [ ]:
#Test the Chat Completions API
llm_response("You're a helpful AI assistant", "Capital of France is")

'The capital of France is Paris.'

### Utility Function

Converts the LLM output into a valid Python dictionary by removing markdown formatting and extracting the JSON content.


In [ ]:
# ──  LLM JSON response ───────────────────────────────────────────
def parse_llm_json(raw: str) -> dict:
    """Strip markdown fences and parse the first JSON object found."""
    # Use regex to remove common markdown code block markers (e.g., ```json)
    cleaned = re.sub(r"```(?:json)?\s*", "", raw).strip().rstrip("`")

    # Locate the first '{' and last '}' to isolate the JSON object if there is extra text
    match = re.search(r"\{.*\}", cleaned, re.DOTALL)
    if match:
        cleaned = match.group(0)

    # Convert the cleaned string into a Python dictionary
    return json.loads(cleaned)

# Evaluation Workflow

## Phase 1 - Data Ingestion

What this phase does:

* Loads the raw sneaker review dataset for the classification task.
* Analyzes the class distribution to understand label balance across categories.
* Splits the dataset into training (80%) and test (20%) sets.
* Reserves the test set exclusively for final evaluation in Phase 7.


Loads the complete sneaker review dataset from a CSV file into a Pandas DataFrame.  

In [ ]:
# ── Load the full dataset ──────────────────────────────────────────────────────
df = pd.read_csv("NimaABSAdataset.csv")

Analyze the distribution of aspect and sentiment labels, along with their combinations, to understand the dataset composition and identify potential class imbalance.

In [ ]:
# ── Inspect class distributions ───────────────────────────────────────────────
print("\n Aspect distribution:")
print(df["aspect"].value_counts().to_string())

print("\n Sentiment distribution:")
print(df["sentiment"].value_counts().to_string())

# Cross-tabulation of aspect × sentiment
print("\n Aspect × Sentiment cross-tab:")
print(pd.crosstab(df["aspect"], df["sentiment"]))


 Aspect distribution:
aspect
durability     29
sizing         27
comfort        26
design         25
price/value    22
color          21

 Sentiment distribution:
sentiment
Negative    79
Positive    59
Neutral     12

 Aspect × Sentiment cross-tab:
sentiment    Negative  Neutral  Positive
aspect                                  
color               7        4        10
comfort            11        1        14
design             10        2        13
durability         23        0         6
price/value         8        4        10
sizing             20        1         6


Split the dataset into training and test sets while preserving aspect label distribution to ensure balanced evaluation.

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.20,         # Reserve 20% data for testing
    stratify=df["aspect"],  # Preserve aspect class distribution in both splits
    random_state=42,        # Ensure reproducible results
)

print(f"Train set : {len(train_df)} reviews")  # Display number of training samples
print(f"Test set  : {len(test_df)} reviews")   # Display number of test samples

Train set : 120 reviews
Test set  : 30 reviews


Display the number of samples in the training and test sets and verify that the aspect label distribution is preserved after splitting.

In [ ]:
print(f"Train set : {len(train_df)} reviews")
print(f"Test set  : {len(test_df)} reviews")

train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print("\Train set aspect distribution:")
print(train_df["aspect"].value_counts().to_string())

print("\nTest set aspect distribution:")
print(test_df["aspect"].value_counts().to_string())

Train set : 120 reviews
Test set  : 30 reviews
\Train set aspect distribution:
aspect
durability     23
comfort        21
sizing         21
design         20
price/value    18
color          17

Test set aspect distribution:
aspect
durability     6
sizing         6
design         5
comfort        5
price/value    4
color          4


## Phase 2 - Configuration Store

Define the initial system prompt that instructs the LLM to classify the sneaker review based on aspect and sentiment.

In [ ]:
ACTIVE_PROMPT_VERSION = "v1.0"

CLASSIFICATION_SYSTEM_PROMPT_v1 = """

Analyse the following customer review for a sneaker and identify:

1. The primary product ASPECT (one out of sizing, comfort, durability, color, design, price/value) mentioned in the review.
2. The SENTIMENT (one out of Positive, Negative, Neutral) expressed towards that aspect.

"""

## Phase 3 - Summarisation

What this phase does:

* Defines the user prompt template used to pass individual reviews to the LLM.
* Creates a helper function to handle LLM-based classification.
* Runs the complete train dataset using Prompt v1.
* Stores both the actual labels and predicted labels together in a DataFrame.


Define the user prompt template that formats each sneaker review and instructs the model to return the prediction strictly in JSON format.

In [ ]:
user_prompt= """Review:

\"{input}\"

Respond ONLY with a JSON object in exactly this format:
{{
  "aspect": "<aspect>",
  "sentiment": "<sentiment>"
}}"""

Create a helper function that sends a review to the LLM, parses the JSON response, and returns fallback labels if parsing

In [ ]:
def classify_review(review_text: str,CLASSIFICATION_SYSTEM_PROMPT,user_prompt) -> dict:
    """
    Run ABSA classification using Prompt v1.
    Returns a dict with keys: aspect, sentiment.
    Falls back to 'unknown' if parsing fails.
    """
    user_prompt = user_prompt.format(input=review_text)
    raw = llm_response(CLASSIFICATION_SYSTEM_PROMPT, user_prompt, temperature=0.0)
    try:
        return parse_llm_json(raw)
    except Exception:
        return {"aspect": "unknown", "sentiment": "unknown"}


Run the training dataset through Prompt v1, generate aspect and sentiment predictions for each review, and store the results alongside the actual labels for evaluation.

In [ ]:
# ── Generate predictions for the full train set ────────────────────────────────
print(f"Classifying {len(train_df)} reviews with Prompt v1 ...")

v1_predictions = []
for i, row in train_df.iterrows():
    pred = classify_review(row["text"],CLASSIFICATION_SYSTEM_PROMPT_v1,user_prompt)
    pred["id"] = row["id"]
    v1_predictions.append(pred)
    if (i + 1) % 20 == 0:
        print(f"  Progress: {i+1}/{len(train_df)}")

train_v1 = train_df.copy()
train_v1["pred_aspect_v1"]    = [p["aspect"]    for p in v1_predictions]
train_v1["pred_sentiment_v1"] = [p["sentiment"] for p in v1_predictions]

print(f"\nDone. Generated {len(v1_predictions)} predictions.")

Classifying 120 reviews with Prompt v1 ...
  Progress: 20/120
  Progress: 40/120
  Progress: 60/120
  Progress: 80/120
  Progress: 100/120
  Progress: 120/120

Done. Generated 120 predictions.


## Phase 4 - Pairwise Comparison


What this phase does:

* Runs Prompt v1 on the same train dataset a second time to measure prediction consistency.
* Evaluates how stable the prompt outputs are across repeated executions.
* Checks whether the model produces the same predictions for identical inputs.
* Identifies prompt sensitivity to minor generation or tokenization variations.
* Uses agreement scores as a stability metric rather than an accuracy metric.


Run Prompt v1 again on the same training dataset to evaluate the consistency and stability of the generated predictions.

In [ ]:
print(f"Classifying {len(train_df)} reviews with Prompt v1 (retest) ...")

v1_retest_predictions = []
for i, row in train_df.iterrows():
    pred = classify_review(row["text"], CLASSIFICATION_SYSTEM_PROMPT_v1, user_prompt)
    pred["id"] = row["id"]
    v1_retest_predictions.append(pred)
    if (i + 1) % 20 == 0:
        print(f"  Progress: {i+1}/{len(train_df)}")

train_v1_retest = train_df.copy()
train_v1_retest["pred_aspect_v1_retest"]    = [p["aspect"]    for p in v1_retest_predictions]
train_v1_retest["pred_sentiment_v1_retest"] = [p["sentiment"] for p in v1_retest_predictions]

print(f"\nDone. Generated {len(v1_retest_predictions)} retest predictions with v1.")

Classifying 120 reviews with Prompt v1 (retest) ...
  Progress: 20/120
  Progress: 40/120
  Progress: 60/120
  Progress: 80/120
  Progress: 100/120
  Progress: 120/120

Done. Generated 120 retest predictions with v1.


Merge the original and retest Prompt v1 predictions to compare prediction consistency across repeated runs.

In [ ]:
# Merge original v1 and retest v1 predictions for comparison
comparison_v1_retest_df = train_v1.merge(
    train_v1_retest[["id", "pred_aspect_v1_retest", "pred_sentiment_v1_retest"]],
    on="id",
    suffixes=("_original", "_retest")
)

Compare the original and retest predictions to identify mismatches and calculate the agreement percentage for aspect and sentiment predictions.

In [ ]:
# Identify differences in aspect
diff_aspect_v1_retest = comparison_v1_retest_df[comparison_v1_retest_df["pred_aspect_v1"] != comparison_v1_retest_df["pred_aspect_v1_retest"]]

diff_sentiment_v1_retest = comparison_v1_retest_df[comparison_v1_retest_df["pred_sentiment_v1"] != comparison_v1_retest_df["pred_sentiment_v1_retest"]]

aspect_agreement_v1_retest = (comparison_v1_retest_df["pred_aspect_v1"] == comparison_v1_retest_df["pred_aspect_v1_retest"]).mean() * 100
sentiment_agreement_v1_retest = (comparison_v1_retest_df["pred_sentiment_v1"] == comparison_v1_retest_df["pred_sentiment_v1_retest"]).mean() * 100

print(f"\nAspect prediction agreement (v1 original vs v1 retest): {aspect_agreement_v1_retest:.2f}%")
print(f"Sentiment prediction agreement (v1 original vs v1 retest): {sentiment_agreement_v1_retest:.2f}%")


Aspect prediction agreement (v1 original vs v1 retest): 98.33%
Sentiment prediction agreement (v1 original vs v1 retest): 99.17%


**Observation**

- The Prompt v1 outputs show very high consistency across repeated runs, with 99.17% agreement for both aspect and sentiment predictions. This indicates that the prompt is stable and produces nearly identical predictions for the same inputs, making it reliable for downstream evaluation and prompt optimization.

## Phase 5 - F1 Score

What this phase does:

* Computes three evaluation metrics by comparing the model predictions with the train labels:

  * **Aspect F1** - Evaluates how well the model identifies the correct product aspect.
  * **Sentiment F1** - Evaluates how accurately the model predicts the sentiment.
  * **Joint F1** - Measures how often both the aspect and sentiment are predicted correctly together.

* Uses **macro-averaged F1 score** so that all classes, including less frequent categories like *durability*, are treated equally during evaluation.

* Prevents common classes such as *comfort* from dominating the overall performance score.

We chooses **F1 score** over accuracy because it provides a better balance between precision and recall, especially for imbalanced datasets. Provides a more reliable evaluation by considering both incorrect positive and incorrect negative predictions.


Compute macro-averaged Aspect F1, Sentiment F1, and Joint F1 scores to evaluate how well Prompt v1 performs on the train dataset.

In [ ]:
# ── F1 Score Evaluation ───────────────────────────────────────────────────────
def evaluate_predictions(df, pred_aspect_col, pred_sentiment_col, label=""):
    """
    Compute Aspect F1, Sentiment F1, and Joint F1 scores using macro averaging.
    """

    # Convert labels to lowercase and remove extra spaces for fair comparison
    train_aspect = df["aspect"].str.lower().str.strip()
    train_sentiment = df["sentiment"].str.lower().str.strip()

    pred_aspect = df[pred_aspect_col].str.lower().str.strip()
    pred_sentiment = df[pred_sentiment_col].str.lower().str.strip()

    # Compute macro F1 score for aspect prediction
    asp_f1 = f1_score(train_aspect, pred_aspect, average="macro", zero_division=0)

    # Compute macro F1 score for sentiment prediction
    sent_f1 = f1_score(train_sentiment, pred_sentiment, average="macro", zero_division=0)

    # Create combined aspect|sentiment labels for joint evaluation
    train_joint = train_aspect + "|" + train_sentiment
    pred_joint = pred_aspect + "|" + pred_sentiment

    # Compute macro F1 score for joint predictions
    joint_f1 = f1_score(train_joint, pred_joint, average="macro", zero_division=0)

    # Display evaluation summary
    print(f"{'='*60}")
    print(f"  Evaluation: {label}")
    print(f"{'='*60}")
    print(f"  Aspect F1 (macro)   : {asp_f1:.3f}")
    print(f"  Sentiment F1 (macro): {sent_f1:.3f}")
    print(f"  Joint F1 (macro)    : {joint_f1:.3f}  (both correct)")

    # Return evaluation metrics as a dictionary
    return {
        "label": label,
        "aspect_f1": asp_f1,
        "sentiment_f1": sent_f1,
        "joint_f1": joint_f1,
    }

# Evaluate Prompt v1 predictions on the training dataset
v1_metrics = evaluate_predictions(
    train_v1,
    "pred_aspect_v1",
    "pred_sentiment_v1",
    label="Prompt v1 - Train Set"
)

  Evaluation: Prompt v1 - Train Set
  Aspect F1 (macro)   : 0.894
  Sentiment F1 (macro): 0.704
  Joint F1 (macro)    : 0.656  (both correct)


## Phase 6 - Prompt Optimisation

In this phase, we move beyond manual prompt writing. Instead of guessing how to improve Prompt v1, we use an automated prompt optimiser to evolve it into a better version, measured against labelled data.

The optimiser we use is called GEPA, which is built into the DeepEval library. Before we configure and run it, let's first understand what GEPA is and how it works.

DeepEval is the optimisation framework we use.
- Inside it, GEPA is the algorithm that automatically evolves the prompt across generations.
- To decide which prompt is better at each step, GEPA needs a way to score outputs - that scorer is GEval.

So: DeepEval runs GEPA, GEPA uses GEval.

### **Why Do We Use DeepEval to Optimise the Prompt?**



So far, Prompt v1 was created manually by writing task instructions, defining valid labels, and specifying the required JSON output format. While this provides a strong starting point, it introduces an important limitation: we do not actually know whether the wording of the prompt is the most effective way to achieve high-quality classifications.

A prompt may appear well-written to a human reader but still produce inconsistent or suboptimal model behaviour. Small wording changes can significantly affect how the LLM interprets instructions, prioritises context, and generates outputs.

DeepEval addresses this problem using **automated prompt optimisation**.

Instead of relying on manual trial-and-error, DeepEval:
1. Takes an initial prompt as the starting point
2. Automatically generates multiple improved prompt variations
3. Evaluates each variation against labelled examples
4. Retains stronger-performing prompts
5. Discards weaker-performing prompts
6. Repeats the optimisation process across multiple generations

The result is a prompt that has been systematically tested against  data and optimised using measurable performance improvements rather than intuition alone.

### **What Does DeepEval Need to Run?**

To optimise prompts effectively, DeepEval requires four key components:

| Component | Purpose |
|---|---|
| **Goldens** | A labelled dataset containing inputs and expected outputs used for evaluation |
| **GEval Metric** | An LLM-as-a-judge evaluation metric that measures prediction quality |
| **model_callback** | A function that executes a candidate prompt and returns the generated response |
| **Optimiser (GEPA)** | The algorithm that generates, scores, and evolves prompt variants across iterations |

Each component plays a specific role in the optimisation pipeline - Goldens provide the data, GEval defines what a good output looks like, model_callback bridges GEPA to the LLM, and GEPA ties everything together to drive the prompt forward.


To learn more about DeepEval and its prompt optimization capabilities, please visit the official website: [DeepEval Official Website](https://www.confident-ai.com/deepeval?utm_source=chatgpt.com)


### DeepEval Golden Dataset

A `Golden` is DeepEval's test-case format each one packages the prompt input and the grounding context the optimiser uses to score outputs. We build one per product from Dataset A.

In [ ]:
goldens = []
for _, row in train_df.iterrows():
    expected = json.dumps({
        "aspect": row["aspect"],
        "sentiment": row["sentiment"]
    })
    goldens.append(Golden(
        input=row["text"],
        expected_output=expected,
    ))

### GEval metrics

We use **GEval**, an LLM-as-a-judge evaluation framework, to measure how well the model predictions match the expected outputs. Unlike simple string matching, GEval uses **Chain-of-Thought (CoT) reasoning** - the evaluator LLM first thinks through its judgement step by step before assigning a score. This makes the evaluation more nuanced and reliable, especially for tasks like ABSA where partial correctness matters.

For this ABSA task, the evaluator checks whether the predicted aspect and sentiment are correct by comparing:

- `INPUT` → Original customer review
- `ACTUAL_OUTPUT` → Model prediction
- `EXPECTED_OUTPUT` → Gold label

The evaluator then **reasons through the comparison** - checking aspect correctness, then sentiment correctness, then deciding on a final score - rather than jumping straight to a number. This CoT process means the score reflects genuine understanding of the prediction quality, not just surface-level text similarity.

**Why and How Do We Tailor GEval?**

Different tasks require different evaluation logic, so GEval can be customized based on project needs. For this ABSA task, the criteria are designed to:
- Reward correct aspect and sentiment predictions  
- Give partial credit if only one prediction is correct  
- Penalize completely incorrect outputs  

We can further tailor GEval by modifying:
- Evaluation criteria  
- Scoring strictness  
- Threshold values  
- Output formatting requirements  

This flexibility allows the evaluation process to align closely with the actual business objective and classification task.

**Key Parameters**

- `criteria` → Defines what makes a good prediction  
- `evaluation_params` → Specifies what the evaluator receives  
- `threshold` → Sets the minimum acceptable score  
- `model` → Defines the evaluator LLM  

This metric will now be used by DeepEval during prompt optimisation.

To know more about G-Eval and how LLM-as-a-judge evaluation works in DeepEval, please refer to the official documentation: [G-Eval Documentation](https://deepeval.com/docs/metrics-llm-evals)

In [ ]:
ABSA_CORRECTNESS_METRIC = GEval(
    name="ABSA Correctness",
    criteria=(
        "Evaluate whether the actual output correctly identifies both the "
        "product ASPECT and SENTIMENT described in the customer review. "
        "Award full marks if aspect AND sentiment both match the expected output. "
        "Partial marks if only one matches. Zero if neither matches."
    ),
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT,
    ],
    threshold=0.7,
    model="gpt-4o-mini",
)

In [ ]:
PROMPT_TEMPLATE_V1 = Prompt(text_template=(CLASSIFICATION_SYSTEM_PROMPT_v1 + user_prompt))

### Model Callback

`model_callback` is the function that tells DeepEval **how to run a prompt and read the result**. At each iteration, the optimizer needs to test a candidate prompt against a product, this function does exactly that: runs the prompt, extracts the classification output, and returns it for scoring.

In [ ]:
def model_callback(prompt: Prompt, golden: Golden) -> str:
    """
    Called by DeepEval at each optimisation iteration with the current
    candidate prompt and a golden. Returns the summary as a plain string
    so GEval metrics can evaluate it directly.
    """
    # Interpolate the current prompt template with input from the golden record
    interpolated = prompt.interpolate(input=golden.input)

    # Call the chat helper with low temperature for deterministic evaluation
    raw = llm_response("", interpolated, temperature=0.1)

    return raw

### GEPA Prompt Optimisation

GEPA stands for **Generative Evolutionary Prompt Adaptation**. It is an automated prompt optimisation algorithm built into DeepEval that improves your prompt without any manual rewriting.

The core idea is simple - instead of you guessing how to make a better prompt, GEPA **generates improved variants automatically, scores them against your labelled data, and keeps the best ones**. It repeats this process across multiple generations, each time building on the survivors of the last.

**How Does It Work?**

GEPA is modelled on evolutionary biology - survival of the fittest, but for prompts.

```
Start → Prompt v1 (your handwritten prompt)
           ↓
    Score on a mini-batch of Goldens
           ↓
    LLM generates mutant child prompts
           ↓
    Score each child → keep the best (Pareto front)
           ↓
    Repeat for N iterations
           ↓
End → Prompt v2 (the best prompt found)
```

Each iteration, weaker prompts are discarded and stronger ones survive to seed the next generation. Over 5 iterations, the prompt gradually evolves into something measurably better than where it started.


To know more about GEPA and how DeepEval performs automated prompt optimization, please refer to the official documentation: [GEPA Prompt Optimization Documentation](https://deepeval.com/docs/prompt-optimization-gepa?utm_source=chatgpt.com)


Initialize the GPT model that DeepEval will use during the prompt optimisation process to generate and evaluate prompt variations.

In [ ]:
optim_model = GPTModel(model="gpt-4o-mini")

**Parameters Explained**

**`iterations=5`**
How many generations of evolution to run. Each iteration scores the current best prompts, generates mutations, and updates the survivors. More iterations = more exploration = potentially better prompt, but more API calls.

**`pareto_size=3`**
How many top-performing prompt candidates to keep alive between generations. Keeping 3 (instead of just 1) preserves diversity, the next generation mutates from 3 different parents rather than converging too early on a single direction.

**`minibatch_size=8`**
How many Goldens are randomly sampled to score each candidate prompt per iteration. Using a mini-batch keeps cost manageable. The tradeoff is that smaller batches introduce some noise, a genuinely better prompt will still win consistently across iterations.

**`random_seed=42`**
Fixes all randomness inside GEPA - which Goldens get sampled, how mutations are initialised. This makes every run fully reproducible. Same seed → same optimised prompt every time, which is essential for debugging and reporting results.

**`tie_breaker="PREFER_CHILD"`**
Decides what happens when a mutant scores equal to its parent. `PREFER_CHILD` keeps the mutant, this biases GEPA toward **exploration**, willing to move to a new prompt even without a clear improvement, hoping something better lies ahead. `PREFER_PARENT` would be more conservative, only moving on a definitive gain.

**`model_callback=model_callback`**
The function that tells GEPA how to actually run a candidate prompt against a Golden and return the output. GEPA is model-agnostic - it doesn't know you are using Claude. The callback is the adapter that bridges GEPA to your specific LLM setup.

**`metrics=[ABSA_CORRECTNESS_METRIC]`**
The scoring function(s) used to judge each candidate prompt's outputs. GEPA uses these scores to decide which prompts survive. Here we use one GEval metric that awards full marks for correct aspect + sentiment, partial marks if only one is correct, and zero if neither matches.

**`optimizer_model=optim_model`**
The LLM that **writes the mutant prompt variants** - separate from the LLM being evaluated. Here `gpt-4o-mini` reads the current prompt and generates improved versions of it. Claude classifies reviews; `gpt-4o-mini` evolves the prompt.

**Note**: Increasing iterations and pareto_size improves exploration and may lead to better prompt optimization, but it also significantly increases runtime, API usage, and overall computational cost because more candidate prompts need to be generated and evaluated across multiple generations.

In [ ]:
# Configure the PromptOptimizer with the GEPA algorithm.
# - iterations: Number of evolutionary steps to perform.
# - pareto_size: Number of top-performing, diverse candidates to track.
# - minibatch_size: Number of golden examples to evaluate per iteration.
# - random_seed: Ensures reproducibility of the mutation/selection process.
# - tie_breaker: Strategy to use when a child prompt has the same score as its parent.

gepa_optimizer = PromptOptimizer(
    algorithm=GEPA(
        iterations=5,
        pareto_size=3,
        minibatch_size=8, # evaluate 8 goldens per iteration
        random_seed=42,
        tie_breaker="PREFER_CHILD",  # breaks ties in favour of the mutation
    ),
    model_callback=model_callback, # The function that executes the prompt and returns output
    metrics=[ABSA_CORRECTNESS_METRIC],      # The list of GEval metrics to optimize for
    optimizer_model=optim_model    # The LLM that generates the optimized prompt variants
)

In [ ]:
# Ensure the algorithm instance specifically uses the provided optimizer model.
gepa_optimizer.algorithm.optimizer_model = optim_model

# Start the optimization process.
# This will iteratively evaluate and mutate the 'PROMPT_TEMPLATE_V1'
# using the provided 'deepeval_goldens' dataset.
gepa_prompt = gepa_optimizer.optimize(
    prompt=PROMPT_TEMPLATE_V1,
    goldens=goldens,
)

Output()

AttributeError: 'str' object has no attribute 'value'

**Original Prompt**

In [ ]:
print(PROMPT_TEMPLATE_V1.text_template)

**GEPA Optimized Prompt**

In [ ]:
print(gepa_prompt.text_template)

**Observation**

The optimized GEPA prompt shows that the optimiser automatically introduced additional rules and constraints into the prompt to improve prediction accuracy and consistency.

In [ ]:
CLASSIFICATION_SYSTEM_PROMPT_v2 = gepa_prompt.text_template

### Validating the Optimised Prompt on Gold Dataset

Run the training dataset through the optimized Prompt v2 and store the generated aspect and sentiment predictions for evaluation and comparison with Prompt v1.

In [ ]:
# ── Generate predictions for the full train set ────────────────────────────────
print(f"Classifying {len(train_df)} reviews with Prompt v2 ...")

v2_predictions = []
for i, row in train_df.iterrows():
    pred = classify_review(row["text"],CLASSIFICATION_SYSTEM_PROMPT_v2,user_prompt)
    pred["id"] = row["id"]
    v2_predictions.append(pred)
    if (i + 1) % 20 == 0:
        print(f"  Progress: {i+1}/{len(train_df)}")

train_v2 = train_df.copy()
train_v2["pred_aspect_v2"]    = [p["aspect"]    for p in v2_predictions]
train_v2["pred_sentiment_v2"] = [p["sentiment"] for p in v2_predictions]

print(f"\nDone. Generated {len(v2_predictions)} predictions.")


In [ ]:
v2_gold_metrics = evaluate_predictions(
    train_v2,
    "pred_aspect_v2",
    "pred_sentiment_v2",
    label="Prompt v2 - Train Set"
)

Compare the evaluation metrics of Prompt v1 and Prompt v2 on the training dataset to measure the improvement achieved through GEPA-based prompt optimization.

In [ ]:
# ── Side-by-side comparison: Prompt v1 vs Prompt v2 ───────────────────────────

# Create a comparison table using evaluation metrics from both prompts
comparison = pd.DataFrame([v1_metrics, v2_gold_metrics])

# Convert F1 scores into percentage format for readability
comparison["aspect_f1_%"] = (comparison["aspect_f1"] * 100).round(1)
comparison["sentiment_f1_%"] = (comparison["sentiment_f1"] * 100).round(1)
comparison["joint_f1_%"] = (comparison["joint_f1"] * 100).round(1)

# Display formatted comparison table
print("\n HEAD-TO-HEAD: Prompt v1 vs Prompt v2 (Train Set)\n")

print(tabulate(
    comparison[["label", "aspect_f1_%", "sentiment_f1_%", "joint_f1_%"]],
    headers=["Prompt", "Aspect F1 %", "Sentiment F1 %", "Joint F1 %"],
    tablefmt="rounded_grid",
    showindex=False,
))

# Calculate performance improvement from Prompt v1 to Prompt v2
delta_asp = v2_gold_metrics["aspect_f1"] - v1_metrics["aspect_f1"]
delta_sent = v2_gold_metrics["sentiment_f1"] - v1_metrics["sentiment_f1"]
delta_jt = v2_gold_metrics["joint_f1"] - v1_metrics["joint_f1"]

# Display metric improvements
print(f"\n🔺 Delta (v2 - v1):")
print(f"   Aspect F1   : {delta_asp:+.3f}")
print(f"   Sentiment F1: {delta_sent:+.3f}")
print(f"   Joint F1    : {delta_jt:+.3f}")

## Phase 7 - Evaluate Optimised Prompt on Test Dataset

What this phase does:

* Run the optimized Prompt v2 on the held-out test set for the first and only time to compute the final Aspect F1, Sentiment F1, and Joint F1 scores on unseen data.

* Evaluate whether the improvements generalize beyond the training distribution; scores close to the gold set indicate good generalization, while significantly lower test scores may indicate overfitting.


Run the optimized Prompt v2 on the unseen test dataset and store the generated aspect and sentiment predictions for final evaluation.

In [ ]:
print(f"Classifying {len(test_df)} reviews with Prompt v2 (TEST set) ...")  # Display total test reviews

v2_test_predictions = []  # Store Prompt v2 test predictions

for i, row in test_df.iterrows():
    # Generate predictions using the optimized Prompt v2
    pred = classify_review(row["text"], CLASSIFICATION_SYSTEM_PROMPT_v2, user_prompt)

    pred["id"] = row["id"]  # Attach review ID to prediction
    v2_test_predictions.append(pred)  # Store prediction result

    # Display progress after every 10 reviews
    if (i + 1) % 10 == 0:
        print(f"  Progress: {i+1}/{len(test_df)}")

# Create a copy of the test dataset to store predictions
test_v2 = test_df.copy()

# Store predicted aspect labels
test_v2["pred_aspect_v2"] = [p["aspect"] for p in v2_test_predictions]

# Store predicted sentiment labels
test_v2["pred_sentiment_v2"] = [p["sentiment"] for p in v2_test_predictions]

# Display completion message
print(f"\nDone - {len(v2_test_predictions)} test predictions generated")

In [ ]:
# ── Evaluate Prompt v2 on test set ────────────────────────────────────────────
v2_test_metrics = evaluate_predictions(
    test_v2,
    "pred_aspect_v2",
    "pred_sentiment_v2",
    label="Prompt v2 - TEST Set (unseen)"
)

In [ ]:
# ── Final 3-way comparison table ──────────────────────────────────────────────
all_metrics = pd.DataFrame([v1_metrics, v2_gold_metrics, v2_test_metrics])
all_metrics["aspect_f1_%"]    = (all_metrics["aspect_f1"]    * 100).round(1)
all_metrics["sentiment_f1_%"] = (all_metrics["sentiment_f1"] * 100).round(1)
all_metrics["joint_f1_%"]     = (all_metrics["joint_f1"]     * 100).round(1)

print("\n FINAL RESULTS SUMMARY\n")
print(tabulate(
    all_metrics[["label","aspect_f1_%","sentiment_f1_%","joint_f1_%"]],
    headers=["Evaluation","Aspect F1 %","Sentiment F1 %","Joint F1 %"],
    tablefmt="rounded_grid",
    showindex=False,
))


# Conclusions

- Prompt optimization using DeepEval GEPA improved the overall classification performance, particularly for sentiment prediction and joint aspect-sentiment understanding.  

- The optimized Prompt v2 achieved very strong performance on the unseen test dataset, indicating that the improvements generalized well beyond the training data.  

<font size=6 color='#4682B4'>Power Ahead!</font>
___